<a href="https://colab.research.google.com/github/emmacotonat/emmacotonat.github.io/blob/main/Algoritmo_de_Newman_Girvan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Instalar paquetes necesarios
!pip install networkx pandas matplotlib openpyxl community

# Importar librerías necesarias
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from community import community_louvain
import itertools

# Definir las rutas de los archivos Excel
ruta_archivo = 'ruta_de_archivo_aquí.xlsx' #Cambiar la ruta del archivo

# Cargar datos desde los archivos Excel
nodos = pd.read_excel(ruta_archivo, sheet_name='Nodes') # Cambiar el tipo de archivo a leer y el nombre de la hoja
enlaces = pd.read_excel(ruta_archivo, sheet_name='Edges') # Cambiar el tipo de archivo a leer y el nombre de la hoja


# Crear el grafo
G = nx.from_pandas_edgelist(enlaces, 'Source', 'Target')  # Cambiar los nombres de las columnas que contienen los nodos de origen y destino

# Añadir todos los nodos, incluyendo los aislados
G.add_nodes_from(nodos['Id'])  # Cambiar el nombre de la columna que contiene los nombres de los nodos, p.ej. Id o ID

# Identificar nodos aislados
nodos_aislados = set([nodo for nodo in G.nodes() if G.degree(nodo) == 0])

def newman_girvan(G, k):
    """
    Implementa el algoritmo de Newman-Girvan para detectar comunidades.

    :param G: Grafo de NetworkX
    :param k: Número deseado de comunidades
    :return: Lista de conjuntos, donde cada conjunto contiene los nodos de una comunidad
    """
    comp = [set(G.nodes())]
    for _ in range(k - 1):
        edges_to_remove = nx.edge_betweenness_centrality(G)
        edge = max(edges_to_remove, key=edges_to_remove.get)
        G.remove_edge(*edge)
        comp = list(nx.connected_components(G))
    return comp

def calcular_modularidad(G, comunidades):
    """
    Calcula la modularidad de una partición de comunidades.

    :param G: Grafo de NetworkX
    :param comunidades: Lista de conjuntos de nodos que representan comunidades
    :return: Valor de modularidad
    """
    particion = {}
    for i, comunidad in enumerate(comunidades):
        for nodo in comunidad:
            particion[nodo] = i
    return community_louvain.modularity(particion, G)

# Rango de números de comunidades a probar
rango_comunidades = range(2, 11)

resultados = []

for k in rango_comunidades:
    G_copy = G.copy()  # Trabajamos con una copia para no modificar el grafo original
    comunidades = newman_girvan(G_copy, k)
    modularidad = calcular_modularidad(G, comunidades)

    # Crear un diccionario que mapea cada nodo a su comunidad
    comunidad_por_nodo = {}
    for i, com in enumerate(comunidades):
        for nodo in com:
            # Asignar 0 si es aislado, de lo contrario asignar i+1
            if nodo in nodos_aislados:
                comunidad_por_nodo[nodo] = 0
            else:
                comunidad_por_nodo[nodo] = i + 1

    resultados.append({
        'num_comunidades': k,
        'modularidad': modularidad,
        'comunidades': comunidad_por_nodo
    })

# Crear un DataFrame con los resultados
df_resultados = pd.DataFrame(resultados)

# Encontrar el número de comunidades con la modularidad más alta
mejor_k = df_resultados.loc[df_resultados['modularidad'].idxmax(), 'num_comunidades']

print(f"El número óptimo de comunidades es {mejor_k}")
print(df_resultados)

# Crear una copia del DataFrame de nodos original
nodos_con_comunidades = nodos.copy()

# Añadir columnas de comunidades al DataFrame de nodos
for resultado in resultados:
    k = resultado['num_comunidades']
    # Mapear cada nodo a su comunidad
    comunidades_dict = resultado['comunidades']
    nodos_con_comunidades[f'Comunidad_k{k}'] = nodos_con_comunidades['Id'].map(comunidades_dict)

# Añadir columna especial para el número óptimo de comunidades
mejor_resultado = next(r for r in resultados if r['num_comunidades'] == mejor_k)
nodos_con_comunidades['Comunidad_Optima'] = nodos_con_comunidades['Id'].map(mejor_resultado['comunidades'])

# Preparar DataFrame de resumen para la hoja de análisis de comunidades
df_resumen_comunidades = df_resultados[['num_comunidades', 'modularidad']].copy()

# Añadir información sobre el tamaño de cada comunidad
tamaños_comunidades = []
for resultado in resultados:
    comunidades_dict = resultado['comunidades']
    tamaño_info = pd.Series(comunidades_dict).value_counts().to_dict()
    tamaños_comunidades.append(tamaño_info)

# Crear columnas para los tamaños de comunidades
max_comunidades = max(df_resumen_comunidades['num_comunidades'])
for i in range(max_comunidades):
    df_resumen_comunidades[f'Tamaño_Comunidad_{i+1}'] = [
        tamaños.get(i+1, 0) for tamaños in tamaños_comunidades
    ]

# Añadir columna para nodos aislados
df_resumen_comunidades['Nodos_Aislados'] = [
    tamaños.get(0, 0) for tamaños in tamaños_comunidades
]

# Exportar a Excel con todas las hojas
nombre_archivo_salida = 'analisis_comunidades_newman_girvan.xlsx'
with pd.ExcelWriter(nombre_archivo_salida, engine='openpyxl') as writer:
    # Hoja 1: Nodes con columnas de comunidades añadidas
    nodos_con_comunidades.to_excel(writer, sheet_name='Nodes', index=False)

    # Hoja 2: Edges (conservada del archivo original)
    enlaces.to_excel(writer, sheet_name='Edges', index=False)

    # Hoja 3: Resumen del análisis de comunidades
    df_resumen_comunidades.to_excel(writer, sheet_name='Analisis_Comunidades', index=False)

files.download(nombre_archivo_salida)

print(f"\n{'='*60}")
print(f"ANÁLISIS COMPLETADO")
print(f"{'='*60}")
print(f"El archivo {nombre_archivo_salida} ha sido creado y está listo para descargar.")
print(f"\nContenido del archivo:")
print(f"  - Hoja 'Nodes': {len(nodos_con_comunidades)} nodos con {len(nodos_con_comunidades.columns)} atributos")
print(f"  - Hoja 'Edges': {len(enlaces)} conexiones")
print(f"  - Hoja 'Analisis_Comunidades': Resumen de modularidad y tamaños")
print(f"\nNúmero óptimo de comunidades: {mejor_k} (modularidad: {df_resultados.loc[df_resultados['num_comunidades']==mejor_k, 'modularidad'].values[0]:.4f})")
print(f"{'='*60}\n")

# Visualizar la modularidad para diferentes números de comunidades
plt.figure(figsize=(12, 6))

# Gráfico principal
plt.plot(df_resultados['num_comunidades'], df_resultados['modularidad'],
         marker='o', linewidth=2, markersize=8, color='steelblue')

# Marcar el óptimo
mejor_modularidad = df_resultados.loc[df_resultados['num_comunidades']==mejor_k, 'modularidad'].values[0]
plt.plot(mejor_k, mejor_modularidad, marker='*', markersize=20,
         color='red', label=f'Óptimo (k={mejor_k})')

plt.xlabel('Número de comunidades', fontsize=12)
plt.ylabel('Modularidad', fontsize=12)
plt.title('Modularidad vs Número de comunidades (Newman-Girvan)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

# Gráfico adicional: Distribución de tamaños de comunidades para el óptimo
plt.figure(figsize=(10, 6))
mejor_tamaños = pd.Series(mejor_resultado['comunidades']).value_counts().sort_index()
colores = ['lightcoral' if idx == 0 else 'steelblue' for idx in mejor_tamaños.index]
etiquetas = ['Aislados' if idx == 0 else f'Com. {idx}' for idx in mejor_tamaños.index]

plt.bar(range(len(mejor_tamaños)), mejor_tamaños.values, color=colores)
plt.xlabel('Comunidad', fontsize=12)
plt.ylabel('Número de nodos', fontsize=12)
plt.title(f'Distribución de nodos por comunidad (k={mejor_k})', fontsize=14, fontweight='bold')
plt.xticks(range(len(mejor_tamaños)), etiquetas, rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()